# NB03: Univariate Phenotype-Metal Associations

Test each phenotype feature individually against metal tolerance score.

- Binary features: Mann-Whitney U, Cohen's d
- Continuous features: Spearman correlation
- Multiple testing correction: BH-FDR
- Phylogenetic control: class/order-level stratification
- Circular reasoning control: partial correlation controlling for n_metal_clusters

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

MAIN_REPO = '/home/psdehal/pangenome_science/BERIL-research-observatory'
PROJ = os.path.join(MAIN_REPO, 'projects', 'bacdive_phenotype_metal_tolerance')
DATA_OUT = os.path.join(PROJ, 'data')
FIG_OUT = os.path.join(PROJ, 'figures')

# Load feature matrix from NB02
feat = pd.read_csv(os.path.join(DATA_OUT, 'species_phenotype_matrix.csv'))
print(f'Feature matrix: {len(feat)} species')

## 1. Binary feature tests

In [ ]:
def cohens_d(group1, group2):
    """Compute Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0.0
    return (group1.mean() - group2.mean()) / pooled_std

binary_features = [
    ('Gram stain (+)', 'gram_positive_binary', 'H1a'),
    ('Motility', 'motile_binary', None),
    ('Catalase (+)', 'catalase_pos_binary', 'H1d'),
    ('Oxidase (+)', 'oxidase_pos_binary', None),
    ('Urease (+)', 'urease_pos_binary', 'H1e'),
    ('Nitrate reduction', 'nitrate_pos_binary', None),
    ('H₂S production', 'h2s_pos_binary', 'H1f'),
    ('Acetate utilization', 'acetate_pos_binary', None),
]

results = []
for name, col, hypothesis in binary_features:
    df = feat[[col, 'metal_score_norm']].dropna()
    if len(df) < 10:
        continue
    pos = df[df[col] == 1]['metal_score_norm']
    neg = df[df[col] == 0]['metal_score_norm']
    if len(pos) < 5 or len(neg) < 5:
        continue
    
    u_stat, p_val = stats.mannwhitneyu(pos, neg, alternative='two-sided')
    d = cohens_d(pos, neg)
    
    results.append({
        'Feature': name,
        'Column': col,
        'Hypothesis': hypothesis or '',
        'n_positive': len(pos),
        'n_negative': len(neg),
        'mean_pos': pos.mean(),
        'mean_neg': neg.mean(),
        'delta': pos.mean() - neg.mean(),
        'cohens_d': d,
        'U_stat': u_stat,
        'p_value': p_val,
        'test': 'Mann-Whitney U',
    })

binary_results = pd.DataFrame(results)
print(f'Binary feature tests: {len(binary_results)}')
print(binary_results[['Feature', 'n_positive', 'n_negative', 'delta', 'cohens_d', 'p_value']].to_string(index=False))

## 2. Continuous feature tests

In [ ]:
continuous_features = [
    ('Metabolite breadth', 'metabolite_frac_pos_mean', 'H1c'),
    ('Enzyme breadth', 'enzyme_frac_pos_mean', None),
]

cont_results = []
for name, col, hypothesis in continuous_features:
    df = feat[[col, 'metal_score_norm']].dropna()
    if len(df) < 10:
        continue
    rho, p_val = stats.spearmanr(df[col], df['metal_score_norm'])
    cont_results.append({
        'Feature': name,
        'Column': col,
        'Hypothesis': hypothesis or '',
        'n': len(df),
        'spearman_rho': rho,
        'p_value': p_val,
        'test': 'Spearman',
    })

cont_df = pd.DataFrame(cont_results)
print(cont_df.to_string(index=False))

## 3. Oxygen tolerance (categorical)

In [ ]:
# Test H1b: anaerobes/facultative vs aerobes
oxy_df = feat[['oxygen_simple_majority', 'metal_score_norm']].dropna()
print(f'Oxygen tolerance test: {len(oxy_df)} species')
print(oxy_df['oxygen_simple_majority'].value_counts().to_string())

if len(oxy_df) >= 10:
    # Kruskal-Wallis across all groups
    groups = [g['metal_score_norm'].values for _, g in oxy_df.groupby('oxygen_simple_majority')]
    if len(groups) >= 2:
        h_stat, kw_p = stats.kruskal(*groups)
        print(f'\nKruskal-Wallis: H={h_stat:.2f}, p={kw_p:.4e}')
    
    # Pairwise: anaerobe vs aerobe
    anaerobes = oxy_df[oxy_df['oxygen_simple_majority'] == 'anaerobe']['metal_score_norm']
    aerobes = oxy_df[oxy_df['oxygen_simple_majority'] == 'aerobe']['metal_score_norm']
    if len(anaerobes) >= 5 and len(aerobes) >= 5:
        u, p = stats.mannwhitneyu(anaerobes, aerobes, alternative='two-sided')
        d = cohens_d(anaerobes, aerobes)
        print(f'Anaerobe vs Aerobe: d={d:.3f}, p={p:.4e}, n_anaerobe={len(anaerobes)}, n_aerobe={len(aerobes)}')
    
    # Per-group means
    print('\nMean metal_score_norm by oxygen group:')
    print(oxy_df.groupby('oxygen_simple_majority')['metal_score_norm'].agg(['mean', 'std', 'count']).to_string())

## 4. Multiple testing correction

In [ ]:
# Combine all p-values for FDR correction
all_results = []
for _, row in binary_results.iterrows():
    all_results.append({
        'Feature': row['Feature'],
        'Hypothesis': row['Hypothesis'],
        'Effect': f"d={row['cohens_d']:.3f}",
        'n': row['n_positive'] + row['n_negative'],
        'p_value': row['p_value'],
    })
for _, row in cont_df.iterrows():
    all_results.append({
        'Feature': row['Feature'],
        'Hypothesis': row['Hypothesis'],
        'Effect': f"rho={row['spearman_rho']:.3f}",
        'n': row['n'],
        'p_value': row['p_value'],
    })

all_df = pd.DataFrame(all_results)

# BH-FDR correction
if len(all_df) > 0:
    reject, q_vals, _, _ = multipletests(all_df['p_value'], method='fdr_bh')
    all_df['q_value'] = q_vals
    all_df['significant'] = reject

    print('=== All Univariate Results (FDR-corrected) ===')
    print(all_df.sort_values('p_value').to_string(index=False))

    all_df.to_csv(os.path.join(DATA_OUT, 'univariate_results.csv'), index=False)
    print(f'\nSaved: data/univariate_results.csv')
    print(f'Significant after FDR: {reject.sum()} / {len(all_df)}')

## 5. Phylogenetic stratification (class-level)

In [ ]:
# Test key features within major classes to control for phylogenetic confounding
key_binary = ['gram_positive_binary', 'catalase_pos_binary', 'urease_pos_binary', 'h2s_pos_binary']

# Get classes with enough species
class_counts = feat['bacdive_class'].value_counts()
major_classes = class_counts[class_counts >= 50].index.tolist()
print(f'Major classes (≥50 species): {len(major_classes)}')
print(class_counts[class_counts >= 50].to_string())

strat_results = []
for col in key_binary:
    for cls in major_classes:
        df = feat[(feat['bacdive_class'] == cls) & feat[col].notna() & feat['metal_score_norm'].notna()]
        pos = df[df[col] == 1]['metal_score_norm']
        neg = df[df[col] == 0]['metal_score_norm']
        if len(pos) >= 5 and len(neg) >= 5:
            u, p = stats.mannwhitneyu(pos, neg, alternative='two-sided')
            d = cohens_d(pos, neg)
            strat_results.append({
                'Feature': col, 'Class': cls,
                'n_pos': len(pos), 'n_neg': len(neg),
                'cohens_d': d, 'p_value': p,
            })

strat_df = pd.DataFrame(strat_results)
if len(strat_df) > 0:
    print('\n=== Class-Stratified Results ===')
    print(strat_df.to_string(index=False))
    strat_df.to_csv(os.path.join(DATA_OUT, 'stratified_results.csv'), index=False)
else:
    print('\nInsufficient data for class-stratified analysis.')

## 6. Partial correlation (controlling for n_metal_clusters)

In [ ]:
# For continuous features: partial Spearman correlation controlling for n_metal_clusters
def partial_spearman(x, y, z):
    """Partial Spearman correlation between x and y, controlling for z."""
    # Rank-transform all variables
    rx = stats.rankdata(x)
    ry = stats.rankdata(y)
    rz = stats.rankdata(z)
    
    # Residualize x and y against z
    from numpy.polynomial.polynomial import polyfit
    coef_x = np.polyfit(rz, rx, 1)
    coef_y = np.polyfit(rz, ry, 1)
    resid_x = rx - np.polyval(coef_x, rz)
    resid_y = ry - np.polyval(coef_y, rz)
    
    return stats.pearsonr(resid_x, resid_y)

print('=== Partial Correlations (controlling for n_metal_clusters) ===')
for name, col, hypothesis in continuous_features:
    df = feat[[col, 'metal_score_norm', 'n_metal_clusters']].dropna()
    if len(df) < 20:
        continue
    r_partial, p_partial = partial_spearman(
        df[col].values, df['metal_score_norm'].values, df['n_metal_clusters'].values
    )
    rho_raw, p_raw = stats.spearmanr(df[col], df['metal_score_norm'])
    print(f'{name}: raw rho={rho_raw:.3f} (p={p_raw:.3e}), partial r={r_partial:.3f} (p={p_partial:.3e})')

## 7. Summary visualization

In [ ]:
# Forest plot of effect sizes
if len(binary_results) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    plot_df = binary_results.sort_values('cohens_d')
    y_pos = range(len(plot_df))
    
    colors = ['coral' if p < 0.05 else 'steelblue' for p in plot_df['p_value']]
    ax.barh(y_pos, plot_df['cohens_d'], color=colors, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df['Feature'])
    ax.set_xlabel("Cohen's d (positive = feature+ has higher metal score)")
    ax.set_title('Phenotype → Metal Tolerance: Univariate Effect Sizes')
    ax.axvline(x=0, color='black', linewidth=0.5)
    
    # Add significance markers
    for i, (_, row) in enumerate(plot_df.iterrows()):
        sig = '***' if row['p_value'] < 0.001 else '**' if row['p_value'] < 0.01 else '*' if row['p_value'] < 0.05 else 'ns'
        ax.text(max(plot_df['cohens_d']) + 0.02, i, sig, va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_OUT, 'univariate_effect_sizes.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/univariate_effect_sizes.png')